# A1 · Three scalers from scratch: fit, transform and the constant column case

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Implement `StandardScaler`, `MinMaxScaler` and `RobustScaler` with the same interface as scikit-learn (`fit`,
`transform`, `fit_transform`, `inverse_transform`), check them against the original versions and handle the edge case
of a constant column.

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns (`age`, `income`, `job_tenure`, `amount`) plus
a fifth **constant** column (7.0) added by hand. The split is 75/25, `X_tr` and `X_te`. Income has a long
tail with a few very high values.

## Educational purpose

Discover that a scaler is an object with **state**: `fit` computes some numbers, `transform` applies them without recomputing them.
Understand how a constant value (zero variance) would break the division, and why `fit_transform` on the test set is a bug.

## Task

1. Write `MyStandardScaler`: `fit` computes `mean_` and `scale_` (standard deviation with `ddof=0`); `transform` computes `(X − mean_) / scale_`. Where `scale_` is 0, replace it with 1 (the constant column becomes all zeros, not NaN).
2. Write `MyMinMaxScaler` (`min_` and `scale_ = max − min`, with 0 replaced by 1; `transform = (X − min_) / scale_`) and `MyRobustScaler` (`center_ = median`, `scale_ = q75 − q25` with `np.percentile`; 0 replaced by 1).
3. Add to each class `fit_transform` and `inverse_transform` (the exact inverse operation).
4. Check that on the training and test data the results coincide with those of `sklearn.preprocessing`.
5. Compute `refit_difference`: the maximum absolute difference between `MyStandardScaler().fit_transform(X_te)` and `scaler_tr.transform(X_te)`, where `scaler_tr` is fitted on `X_tr`. It is the measure of how much the wrong code changes the data.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

df = ds.loans(with_missing=False)
X_num = df[["age", "income", "job_tenure", "amount"]].to_numpy()
X_num = np.column_stack([X_num, np.full(len(X_num), 7.0)])         # a constant column: the edge case
X_tr, X_te = train_test_split(X_num, test_size=0.25, random_state=0)

## Your work

In [ ]:
class MyStandardScaler:
    def fit(self, X):
        ...
        return self

    def transform(self, X):
        ...

    def fit_transform(self, X):
        ...

    def inverse_transform(self, Z):
        ...

In [ ]:
class MyMinMaxScaler:
    ...


class MyRobustScaler:
    ...

In [ ]:
scaler_tr = MyStandardScaler().fit(X_tr)
refit_difference = ...
refit_difference

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='To neutralize the zero scale: `scale = np.where(scale == 0, 1.0, scale)`. For the Robust: `q25, q75 = np.percentile(X, [25, 75], axis=0)`. The inverse of the standard one is `Z * scale_ + mean_`.'):
    pairs = [(MyStandardScaler, StandardScaler), (MyMinMaxScaler, MinMaxScaler), (MyRobustScaler, RobustScaler)]
    for Mine, Ref in pairs:
        mine, ref = Mine(), Ref()
        check(close(mine.fit_transform(X_tr), ref.fit_transform(X_tr), atol=1e-9),
              f"{Mine.__name__}: fit_transform matches scikit-learn on the training set",
              f"{Mine.__name__}.fit_transform(X_tr) does not match {Ref.__name__}")
        check(close(mine.transform(X_te), ref.transform(X_te), atol=1e-9),
              f"{Mine.__name__}: transform uses the training statistics",
              f"{Mine.__name__}.transform(X_te) does not match: did you recompute the statistics on the test set?")
        check(close(mine.inverse_transform(mine.transform(X_te)), X_te, atol=1e-6),
              f"{Mine.__name__}: inverse_transform rebuilds the data", "transform followed by inverse_transform must give back X")
        check(not np.isnan(mine.transform(X_te)).any() and np.all(mine.transform(X_te)[:, 4] == 0),
              f"{Mine.__name__}: the constant column becomes zero, with no NaN", "where the scale is 0 use 1")
    s = MyStandardScaler().fit(X_tr)
    expected = np.abs(MyStandardScaler().fit_transform(X_te) - s.transform(X_te)).max()
    check(close(refit_difference, expected) and refit_difference > 0.1,
          "refitting the scaler on the test set changes the data substantially",
          "refit_difference = max |fit_transform(X_te) - scaler_tr.transform(X_te)|")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_scalers_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.